# Chapter 16: 워크플로우 아키텍처 패턴 (Workflow Architecture Patterns)

LLM 호출을 체계적으로 조합하는 5가지 핵심 패턴을 실습합니다.

**Sections:**
- 16.0 Setup & Environment
- 16.1 Prompt Chaining (순차 체이닝)
- 16.2 Prompt Chaining + Gate (조건부 게이트)
- 16.3 Routing (동적 라우팅)
- 16.4 Parallelization (병렬 실행)
- 16.5 Orchestrator-Workers (동적 작업 분배)

---
## 16.0 Setup & Environment

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

print("OPENAI_API_KEY:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("OPENAI_BASE_URL:", os.getenv("OPENAI_BASE_URL"))
print("OPENAI_MODEL_NAME:", os.getenv("OPENAI_MODEL_NAME"))

OPENAI_API_KEY: a703ad29...
OPENAI_BASE_URL: https://lgcns-assetization3-japan-east.openai.azure.com/openai/v1
OPENAI_MODEL_NAME: gpt-5.1


In [2]:
from importlib.metadata import version

print(f"langgraph: {version('langgraph')}")
print(f"langchain: {version('langchain')}")

langgraph: 1.1.3
langchain: 1.2.13


---
## 16.1 Prompt Chaining — 순차적 체이닝

여러 LLM 호출을 **순차적으로 연결** — 이전 단계의 출력이 다음 단계의 입력:

```
START → list_ingredients → create_recipe → describe_plating → END
```

핵심: `with_structured_output()`으로 LLM 응답을 구조화된 형태로 강제

In [3]:
import os
from typing import List
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel

llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")


# 상태 정의
class State(TypedDict):
    dish: str
    ingredients: list[dict]
    recipe_steps: str
    plating_instructions: str


# 구조화된 출력 스키마
class Ingredient(BaseModel):
    name: str
    quantity: str
    unit: str


class IngredientsOutput(BaseModel):
    ingredients: List[Ingredient]


# 노드 함수들
def list_ingredients(state: State):
    structured_llm = llm.with_structured_output(IngredientsOutput)
    response = structured_llm.invoke(
        f"List 5-8 ingredients needed to make {state['dish']}"
    )
    return {"ingredients": [i.model_dump() for i in response.ingredients]}


def create_recipe(state: State):
    response = llm.invoke(
        f"Write a step by step cooking instruction for {state['dish']}, "
        f"using these ingredients {state['ingredients']}"
    )
    return {"recipe_steps": response.content}


def describe_plating(state: State):
    response = llm.invoke(
        f"Describe how to beautifully plate this dish {state['dish']} "
        f"based on this recipe {state['recipe_steps']}"
    )
    return {"plating_instructions": response.content}


# 그래프 구성
graph_builder = StateGraph(State)

graph_builder.add_node("list_ingredients", list_ingredients)
graph_builder.add_node("create_recipe", create_recipe)
graph_builder.add_node("describe_plating", describe_plating)

graph_builder.add_edge(START, "list_ingredients")
graph_builder.add_edge("list_ingredients", "create_recipe")
graph_builder.add_edge("create_recipe", "describe_plating")
graph_builder.add_edge("describe_plating", END)

graph = graph_builder.compile()

result = graph.invoke({"dish": "hummus"})

print("=== Ingredients ===")
for i in result["ingredients"]:
    print(f"  {i['name']}: {i['quantity']} {i['unit']}")
print("\n=== Recipe (first 300 chars) ===")
print(result["recipe_steps"][:300])
print("\n=== Plating (first 300 chars) ===")
print(result["plating_instructions"][:300])

=== Ingredients ===
  Chickpeas (cooked or canned): 1.5 cups
  Tahini: 1/4 cup
  Olive oil: 2 tablespoons
  Lemon juice (fresh): 2 tablespoons
  Garlic cloves: 2 cloves
  Salt: 1/2 teaspoon
  Ground cumin: 1/2 teaspoon
  Water (or aquafaba from chickpeas): 2-4 tablespoons

=== Recipe (first 300 chars) ===
1. **Prepare the chickpeas**  
   - If using canned: drain and rinse **1½ cups** chickpeas.  
   - For extra-smooth hummus, optionally pinch off and discard some of the skins (not required, but improves texture).

2. **Set up your equipment**  
   - Use a food processor or a blender.  
   - Add to t

=== Plating (first 300 chars) ===
Here’s a simple way to make that hummus look restaurant-level beautiful, using what you likely already have.

---

### 1. Choose the right serving dish  
- Use a **shallow, wide bowl or plate** rather than a deep one. This lets you create swirls and space for toppings.  
- White or light-colored dis


### Exercise 16.1

1. `dish` 값을 바꿔서 다양한 요리에 대한 결과를 비교해 보라.
2. `with_structured_output()`을 사용한 노드와 일반 `invoke()`를 사용한 노드의 차이를 이해하라.
3. 4단계를 추가해 보라 (예: 와인 페어링 추천).

In [4]:
# Try it yourself


---
## 16.2 Prompt Chaining + Gate — 조건부 게이트

LLM 출력이 품질 기준을 충족하지 않으면 **이전 단계를 재실행**:

```
START → list_ingredients → [gate: 3~8개?] → Yes → create_recipe → ...
                              ↑               → No ─┘ (재시도)
```

In [5]:
import os
from typing import List
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel

llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")


class State(TypedDict):
    dish: str
    ingredients: list[dict]
    recipe_steps: str
    plating_instructions: str


class Ingredient(BaseModel):
    name: str
    quantity: str
    unit: str


class IngredientsOutput(BaseModel):
    ingredients: List[Ingredient]


def list_ingredients(state: State):
    structured_llm = llm.with_structured_output(IngredientsOutput)
    response = structured_llm.invoke(
        f"List 5-8 ingredients needed to make {state['dish']}"
    )
    ingredients = [i.model_dump() for i in response.ingredients]
    print(f"  Generated {len(ingredients)} ingredients")
    return {"ingredients": ingredients}


def create_recipe(state: State):
    response = llm.invoke(
        f"Write a step by step cooking instruction for {state['dish']}, "
        f"using these ingredients {state['ingredients']}"
    )
    return {"recipe_steps": response.content}


def describe_plating(state: State):
    response = llm.invoke(
        f"Describe how to beautifully plate this dish {state['dish']} "
        f"based on this recipe {state['recipe_steps']}"
    )
    return {"plating_instructions": response.content}


# 게이트 함수 — 재료 개수 검증
def gate(state: State):
    ingredients = state["ingredients"]
    count = len(ingredients)
    if count > 8 or count < 3:
        print(f"  GATE FAIL: {count} ingredients (need 3-8). Retrying...")
        return False
    print(f"  GATE PASS: {count} ingredients")
    return True


graph_builder = StateGraph(State)

graph_builder.add_node("list_ingredients", list_ingredients)
graph_builder.add_node("create_recipe", create_recipe)
graph_builder.add_node("describe_plating", describe_plating)

graph_builder.add_edge(START, "list_ingredients")
graph_builder.add_conditional_edges(
    "list_ingredients",
    gate,
    {
        True: "create_recipe",       # 통과 → 다음 단계
        False: "list_ingredients",    # 실패 → 재시도
    },
)
graph_builder.add_edge("create_recipe", "describe_plating")
graph_builder.add_edge("describe_plating", END)

graph = graph_builder.compile()

result = graph.invoke({"dish": "bibimbap"})
print(f"\nFinal: {len(result['ingredients'])} ingredients")

  Generated 8 ingredients
  GATE PASS: 8 ingredients



Final: 8 ingredients


### Exercise 16.2

1. 게이트 조건을 `len(ingredients) == 5`로 엄격하게 바꿔서 재시도 횟수를 관찰하라.
2. 무한 루프 방지를 위해 `retry_count` 필드를 State에 추가하고 최대 3회로 제한해 보라.
3. 게이트 패턴이 유용한 시나리오: 코드 구문 검사, 번역 언어 확인, 필수 필드 검증 등.

In [6]:
# Try it yourself


---
## 16.3 Routing — 동적 라우팅

질문 난이도를 LLM이 평가 → 난이도별 다른 모델로 라우팅:

```
START → assess_difficulty → easy   → dumb_node (GPT-3.5)
                          → medium → average_node (GPT-4o)
                          → hard   → smart_node (GPT-5)
```

핵심: `Command` + `Structured Output(Literal)` 로 안전한 라우팅

In [7]:
import os
from typing import Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langchain.chat_models import init_chat_model
from pydantic import BaseModel

llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")

# 실무에서는 비용/성능별 다른 모델 사용
# 여기서는 동일 모델로 시뮬레이션
dumb_llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")
average_llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")
smart_llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")


class State(TypedDict):
    question: str
    difficulty: str
    answer: str
    model_used: str


# Literal로 LLM 응답을 3개 값으로 제한
class DifficultyResponse(BaseModel):
    difficulty_level: Literal["easy", "medium", "hard"]


def assess_difficulty(state: State):
    structured_llm = llm.with_structured_output(DifficultyResponse)
    response = structured_llm.invoke(
        f"""
        Assess the difficulty of this question:
        Question: {state['question']}

        - EASY: Simple facts, basic definitions, yes/no answers
        - MEDIUM: Requires explanation, comparison, analysis
        - HARD: Complex reasoning, multiple steps, deep expertise
        """
    )
    level = response.difficulty_level
    goto_map = {"easy": "dumb_node", "medium": "average_node", "hard": "smart_node"}
    print(f"  Difficulty: {level} → {goto_map[level]}")
    return Command(goto=goto_map[level], update={"difficulty": level})


def dumb_node(state: State):
    response = dumb_llm.invoke(state["question"])
    return {"answer": response.content, "model_used": "gpt-3.5 (simulated)"}


def average_node(state: State):
    response = average_llm.invoke(state["question"])
    return {"answer": response.content, "model_used": "gpt-4o (simulated)"}


def smart_node(state: State):
    response = smart_llm.invoke(state["question"])
    return {"answer": response.content, "model_used": "gpt-5 (simulated)"}


graph_builder = StateGraph(State)

graph_builder.add_node(
    "assess_difficulty", assess_difficulty,
    destinations=("dumb_node", "average_node", "smart_node"),
)
graph_builder.add_node("dumb_node", dumb_node)
graph_builder.add_node("average_node", average_node)
graph_builder.add_node("smart_node", smart_node)

graph_builder.add_edge(START, "assess_difficulty")
graph_builder.add_edge("dumb_node", END)
graph_builder.add_edge("average_node", END)
graph_builder.add_edge("smart_node", END)

graph = graph_builder.compile()

In [8]:
# 쉬운 질문
result = graph.invoke({"question": "What is the capital of France?"})
print(f"Model: {result['model_used']}")
print(f"Answer: {result['answer'][:200]}")

  Difficulty: easy → dumb_node


Model: gpt-3.5 (simulated)
Answer: The capital of France is Paris.


In [9]:
# 어려운 질문
result = graph.invoke({"question": "Explain the economic implications of quantum computing on global supply chains"})
print(f"Model: {result['model_used']}")
print(f"Answer: {result['answer'][:300]}")

  Difficulty: hard → smart_node


Model: gpt-5 (simulated)
Answer: Quantum computing could reshape global supply chains in three main ways: how they’re designed, how risks are managed, and who holds economic power.

---

## 1. Optimization and efficiency gains

Many core supply chain problems are optimization problems: routing, inventory placement, production sched


### Exercise 16.3

1. 다양한 난이도의 질문을 입력하여 라우팅이 올바르게 작동하는지 확인하라.
2. `model_used` 필드를 확인하여 실제로 어떤 경로로 갔는지 검증하라.
3. 모델 선택 외에도, 다른 프롬프트 템플릿으로 라우팅하는 시나리오를 설계해 보라.

In [10]:
# Try it yourself


---
## 16.4 Parallelization — 병렬 실행

독립적인 LLM 호출을 **동시에 병렬 실행** → 모두 끝나면 집계:

```
        → get_summary ────────┐
        → get_sentiment ──────┤
START → → get_key_points ─────┤→ get_final_analysis → END
        → get_recommendation ─┘
```

핵심: `START`에서 여러 노드로 엣지 = 자동 병렬, 여러 노드에서 하나로 = join

In [11]:
import os
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model

llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")


class State(TypedDict):
    document: str
    summary: str
    sentiment: str
    key_points: str
    recommendation: str
    final_analysis: str


# 4개의 병렬 분석 노드 — 같은 document를 읽고, 다른 필드에 쓴다
def get_summary(state: State):
    print("  [parallel] get_summary started")
    response = llm.invoke(f"Write a 3-sentence summary of this document: {state['document'][:500]}")
    return {"summary": response.content}


def get_sentiment(state: State):
    print("  [parallel] get_sentiment started")
    response = llm.invoke(f"Analyse the sentiment and tone of this document: {state['document'][:500]}")
    return {"sentiment": response.content}


def get_key_points(state: State):
    print("  [parallel] get_key_points started")
    response = llm.invoke(f"List the 5 most important points of this document: {state['document'][:500]}")
    return {"key_points": response.content}


def get_recommendation(state: State):
    print("  [parallel] get_recommendation started")
    response = llm.invoke(f"Based on the document, list 3 recommended next steps: {state['document'][:500]}")
    return {"recommendation": response.content}


# 집계 노드 — 병렬 노드들이 모두 완료된 후 실행
def get_final_analysis(state: State):
    print("  [join] get_final_analysis started")
    response = llm.invoke(
        f"""
    Give me a brief analysis combining these results:

    SUMMARY: {state['summary']}
    SENTIMENT: {state['sentiment']}
    KEY POINTS: {state['key_points']}
    RECOMMENDATIONS: {state['recommendation']}
    """
    )
    return {"final_analysis": response.content}


graph_builder = StateGraph(State)

graph_builder.add_node("get_summary", get_summary)
graph_builder.add_node("get_sentiment", get_sentiment)
graph_builder.add_node("get_key_points", get_key_points)
graph_builder.add_node("get_recommendation", get_recommendation)
graph_builder.add_node("get_final_analysis", get_final_analysis)

# START에서 4개 노드로 = 병렬!
graph_builder.add_edge(START, "get_summary")
graph_builder.add_edge(START, "get_sentiment")
graph_builder.add_edge(START, "get_key_points")
graph_builder.add_edge(START, "get_recommendation")

# 4개 모두 → 하나로 = join (모두 끝날 때까지 대기)
graph_builder.add_edge("get_summary", "get_final_analysis")
graph_builder.add_edge("get_sentiment", "get_final_analysis")
graph_builder.add_edge("get_key_points", "get_final_analysis")
graph_builder.add_edge("get_recommendation", "get_final_analysis")

graph_builder.add_edge("get_final_analysis", END)

graph = graph_builder.compile()

In [12]:
# 샘플 문서로 실행
sample_doc = """
The Federal Reserve announced today that it will maintain interest rates at their current level,
citing concerns about inflation remaining above the 2% target. Chair Powell noted that while the
labor market remains strong with unemployment at 3.7%, there are signs of cooling in the housing
sector. The committee emphasized a data-dependent approach to future policy decisions and indicated
that rate cuts could begin later this year if inflation continues to moderate. Markets reacted
positively, with the S&P 500 rising 1.2% following the announcement.
"""

result = graph.invoke({"document": sample_doc})

print("=== Final Analysis (first 500 chars) ===")
print(result["final_analysis"][:500])

  [parallel] get_key_points started
  [parallel] get_recommendation started
  [parallel] get_sentiment started
  [parallel] get_summary started


  [join] get_final_analysis started


=== Final Analysis (first 500 chars) ===
The Fed’s decision and communication point to a “wait-and-see” stance with cautious optimism:

- **Policy stance:** Rates stay unchanged because inflation is still above target, but the Fed is openly considering cuts later this year if data cooperate. This makes the outlook less hawkish than before and explains the mildly positive market reaction.
- **Economic backdrop:** Strong labor market conditions (low unemployment) are offset by cooling in housing, reinforcing a balanced, risk-aware tone r


### Exercise 16.4

1. 5번째 병렬 노드(예: `get_risks`)를 추가하고 결과가 집계에 포함되는지 확인하라.
2. `graph.stream()`으로 실행하여 병렬 노드들이 동시에 시작되는 것을 확인하라.
3. 순차 실행(Chaining)과 병렬 실행의 총 실행 시간을 비교해 보라.

In [13]:
# Try it yourself


---
## 16.5 Orchestrator-Workers — 동적 작업 분배

오케스트레이터가 작업을 **동적으로 분할** → Send API로 워커에게 분배 → 결과 합침:

```
START → orchestrator → [Send × N] → worker × N → synthesizer → END
```

Chapter 13의 Send API + 16.4 Parallelization의 실전 결합

In [14]:
import os
import operator
from typing import Annotated, Union
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain.chat_models import init_chat_model

llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")


class State(TypedDict):
    topic: str
    sections: list[str]            # 오케스트레이터가 분할한 섹션 목록
    results: Annotated[list[dict], operator.add]  # 워커 결과 누적
    final_report: str


# 오케스트레이터: 주제를 분석하여 섹션 목록 생성
def orchestrator(state: State):
    from pydantic import BaseModel
    from typing import List

    class Sections(BaseModel):
        sections: List[str]

    structured_llm = llm.with_structured_output(Sections)
    response = structured_llm.invoke(
        f"Break down this topic into 3-5 research sections: {state['topic']}"
    )
    print(f"  Orchestrator: {len(response.sections)} sections")
    for s in response.sections:
        print(f"    - {s}")
    return {"sections": response.sections}


# 워커: 개별 섹션을 처리 (Send API로 호출됨)
def worker(section: str):
    response = llm.invoke(f"Write a brief paragraph about: {section}")
    print(f"  Worker done: {section[:40]}...")
    return {"results": [{"section": section, "content": response.content}]}


# 디스패처: Send API로 워커를 병렬 실행
def dispatch_workers(state: State):
    return [Send("worker", section) for section in state["sections"]]


# 종합: 모든 워커 결과를 합쳐서 최종 보고서 생성
def synthesizer(state: State):
    sections_text = "\n\n".join(
        f"### {r['section']}\n{r['content']}" for r in state["results"]
    )
    response = llm.invoke(
        f"Combine these sections into a cohesive report:\n\n{sections_text}"
    )
    return {"final_report": response.content}


graph_builder = StateGraph(State)

graph_builder.add_node("orchestrator", orchestrator)
graph_builder.add_node("worker", worker)
graph_builder.add_node("synthesizer", synthesizer)

graph_builder.add_edge(START, "orchestrator")
graph_builder.add_conditional_edges("orchestrator", dispatch_workers, ["worker"])
graph_builder.add_edge("worker", "synthesizer")
graph_builder.add_edge("synthesizer", END)

graph = graph_builder.compile()

result = graph.invoke({"topic": "The impact of AI on healthcare"})

print("\n=== Final Report (first 500 chars) ===")
print(result["final_report"][:500])

  Orchestrator: 5 sections
    - Clinical Applications and Care Delivery
- Diagnostic support (medical imaging, pathology, triage)
- Predictive analytics for risk stratification and early intervention
- Personalized treatment planning and precision medicine
- Virtual health assistants and telehealth integration
- AI in surgery and robotics
    - Health System Operations and Efficiency
- Workflow automation (documentation, scheduling, coding)
- Resource allocation, staffing, and bed management
- Supply chain and inventory optimization
- Reducing administrative burden and burnout
- Cost-effectiveness and economic impact analyses
    - Data, Algorithms, and Infrastructure
- Quality and interoperability of health data (EHRs, registries, wearables)
- Model development, validation, and generalizability
- Bias, fairness, and performance across diverse populations
- Real-world deployment, monitoring, and model updating
- Technical infrastructure and integration with legacy systems
    - Ethica

  Worker done: Clinical Applications and Care Delivery
...
  Worker done: Health System Operations and Efficiency
...


  Worker done: Data, Algorithms, and Infrastructure
- Q...
  Worker done: Public Health, Equity, and Global Implic...


  Worker done: Ethical, Legal, and Regulatory Considera...



=== Final Report (first 500 chars) ===
## AI in Health Care: A Cohesive Overview

AI is reshaping health care from bedside decision‑making to global public health. Its impact can be understood across five interconnected domains: clinical applications, health system operations, data and infrastructure, ethics and regulation, and public health and equity.

---

### 1. Clinical Applications and Care Delivery

AI is transforming clinical care across the continuum—from prevention and diagnosis to treatment and follow‑up.

- **Diagnostic s


### Exercise 16.5

1. `topic`을 바꿔서 LLM이 다른 섹션 분할을 하는지 확인하라.
2. 워커에 `time.sleep(1)`을 추가하여 병렬 실행의 효과를 체감하라.
3. 워커가 구조화된 출력(`BaseModel`)을 반환하도록 수정해 보라.

In [15]:
# Try it yourself


---
## Final Exercises — 종합 실습

### 과제 1: 번역 체인 + 게이트 (★★☆)

한국어 → 영어 → 일본어 순차 번역 체인.
게이트: 번역 결과가 50자 이상이면 통과, 아니면 재시도.

In [16]:
# 과제 1: 여기에 작성


### 과제 2: 감정 기반 라우팅 (★★☆)

사용자 메시지의 감정을 분석 → 감정별 다른 응답 생성:
- positive → 축하 메시지
- negative → 위로 메시지
- neutral → 정보 제공 메시지

In [17]:
# 과제 2: 여기에 작성


### 과제 3: 코드 리뷰 병렬 분석 (★★★)

코드를 입력으로 받아 4가지 관점에서 동시 분석:
- 보안 취약점
- 성능 이슈
- 가독성
- 테스트 커버리지 제안

4개 결과를 합쳐서 종합 리뷰 생성

In [18]:
# 과제 3: 여기에 작성


### 과제 4: Orchestrator-Workers 블로그 작성 (★★★)

주제를 입력하면:
1. 오케스트레이터가 목차(3~5개 섹션)를 생성
2. Send API로 각 섹션을 워커에게 분배
3. Synthesizer가 결과를 합쳐 완성된 블로그 포스트 출력

In [19]:
# 과제 4: 여기에 작성
